# AI Portfolio Analyzer — Colab Training Notebook

Run this notebook on Google Colab (free GPU) to train the LSTM / Transformer model.

Steps:
1. Mount Google Drive
2. Clone / copy project
3. Install dependencies
4. Login to Weights & Biases (optional)
5. Configure and train
6. Save checkpoint to Drive

> **Educational use only — not financial advice.**

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU detected. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ai-portfolio-analyzer'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

In [ ]:
# ── Cell 3: Install Dependencies ───────────────────────────────────────────
# This takes ~3-5 minutes on first run; subsequent runs use cached packages.
!pip install -q \
    yfinance pandas numpy scikit-learn \
    torch torchvision \
    xgboost lightgbm \
    transformers accelerate sentencepiece \
    cvxpy scipy \
    plotly streamlit \
    wandb python-dotenv \
    requests feedparser pyyaml

print('All packages installed.')

In [ ]:
# ── Cell 4: Copy project to /content (for fast I/O) ───────────────────────
import shutil, sys

CONTENT_DIR = '/content/ai-portfolio-analyzer'

# If already in /content, skip copy
if not os.path.exists(CONTENT_DIR):
    shutil.copytree(PROJECT_DIR, CONTENT_DIR)
    print(f'Copied project to {CONTENT_DIR}')
else:
    print(f'Project already at {CONTENT_DIR}')

os.chdir(CONTENT_DIR)
sys.path.insert(0, CONTENT_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 5: W&B Login (optional) ───────────────────────────────────────────
USE_WANDB = False   # ← set True if you have a W&B account

if USE_WANDB:
    import wandb
    wandb.login()   # will prompt for API key
else:
    print('Skipping W&B — set USE_WANDB=True to enable experiment tracking.')

In [ ]:
# ── Cell 6: Load & Patch Config ────────────────────────────────────────────
from src.utils import load_config, set_seed, detect_device

cfg = load_config('configs/config.yaml')

# Override settings for Colab
cfg['model']['type']             = 'lstm'    # or 'transformer'
cfg['model']['hidden_size']      = 128
cfg['model']['num_layers']       = 2
cfg['training']['epochs']        = 60
cfg['training']['batch_size']    = 128
cfg['training']['learning_rate'] = 0.001
cfg['training']['checkpoint_dir'] = 'checkpoints'
cfg['data']['raw_dir']           = 'data/raw'
cfg['data']['processed_dir']     = 'data/processed'
cfg['wandb']['enabled']          = USE_WANDB

# Create dirs
for d in ['data/raw', 'data/processed', 'checkpoints', 'reports/figures']:
    os.makedirs(d, exist_ok=True)

device = detect_device()
set_seed(42)
print(f'Device: {device}')
print(f'Model : {cfg["model"]["type"]}')

In [ ]:
# ── Cell 7: Download Market Data ───────────────────────────────────────────
from src.dataset import download_all

data = download_all(cfg)
print(f'Downloaded {len(data)} tickers:')
for t, df in data.items():
    print(f'  {t:6s}  {len(df):5d} rows  {df.index[0].date()} → {df.index[-1].date()}')

In [ ]:
# ── Cell 8: Build Processed Features ──────────────────────────────────────
from src.dataset import build_processed_dataset

TRAIN_TICKER = 'AAPL'   # ← change to any ticker from cfg['data']['tickers']

df_proc = build_processed_dataset(TRAIN_TICKER, cfg, sentiment_df=None)
print(f'Processed dataset: {df_proc.shape}')
print(df_proc.describe().T[['mean','std','min','max']])

In [ ]:
# ── Cell 9: Train Model ────────────────────────────────────────────────────
# This runs the full training pipeline from src/train.py
from src.train import train

print(f'Training {cfg["model"]["type"].upper()} on {TRAIN_TICKER} ...')
train(cfg, ticker=TRAIN_TICKER)
print('Training complete!')

In [ ]:
# ── Cell 10: Evaluate Model ────────────────────────────────────────────────
from src.evaluate import evaluate

metrics, y_pred, y_true = evaluate(cfg, ticker=TRAIN_TICKER)

print('\nTest Set Metrics:')
for k, v in sorted(metrics.items()):
    print(f'  {k:30s}: {v:.4f}')

In [ ]:
# ── Cell 11: Quick Prediction Sanity Check ─────────────────────────────────
import numpy as np
import plotly.graph_objects as go

n = min(100, len(y_true))
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_true[:n, 0], name='True Return', line=dict(color='#8b949e')))
fig.add_trace(go.Scatter(y=y_pred[:n, 0], name='Pred Return', line=dict(color='#00d4aa')))
fig.update_layout(
    title=f'{TRAIN_TICKER} — Return Prediction (first {n} test samples)',
    template='plotly_dark', height=380,
    xaxis_title='Sample', yaxis_title='Log Return',
)
fig.show()

In [ ]:
# ── Cell 12: Run Live Inference ────────────────────────────────────────────
from src.inference import run_inference

result = run_inference(TRAIN_TICKER, cfg)

print(f"Ticker          : {result['ticker']}")
print(f"Current Price   : ${result.get('current_price', 0):.2f}")
print(f"Predicted Return: {result.get('pred_return', 0)*100:+.2f}%")
print(f"Predicted Vol   : {result.get('pred_volatility', 0)*100:.2f}%")
print(f"Downside Prob   : {result.get('pred_downside_prob', 0)*100:.1f}%")
print(f"Risk Score      : {result.get('risk_score', 0):.0f} / 100  [{result.get('risk_label')}]")
print(f"Outlook         : {result.get('outlook')}")
sf = result.get('sentiment_features', {})
print(f"Sentiment       : {sf.get('weighted_sentiment', 0):+.3f}  ({sf.get('news_volume', 0)} articles)")

In [ ]:
# ── Cell 13: Run Backtest ──────────────────────────────────────────────────
from src.backtest import run_backtest

bt_result = run_backtest(cfg)
metrics_bt = bt_result['metrics']

print('Backtest Results:')
print(f"  CAGR         : {metrics_bt.get('cagr', 0)*100:.2f}%")
print(f"  Sharpe Ratio : {metrics_bt.get('sharpe_ratio', 0):.3f}")
print(f"  Sortino Ratio: {metrics_bt.get('sortino_ratio', 0):.3f}")
print(f"  Max Drawdown : {metrics_bt.get('max_drawdown', 0)*100:.2f}%")
print(f"  Win Rate     : {metrics_bt.get('win_rate', 0)*100:.1f}%")
if 'benchmark_cagr' in metrics_bt:
    print(f"  Benchmark CAGR: {metrics_bt['benchmark_cagr']*100:.2f}%")

In [ ]:
# ── Cell 14: Plot Equity Curve ─────────────────────────────────────────────
from dashboard.dashboard_utils import backtest_chart

ec = bt_result.get('equity_curve')
bm = bt_result.get('benchmark_curve')

if ec is not None:
    fig = backtest_chart(ec, bm)
    if fig:
        fig.show()
else:
    print('No equity curve available.')

In [ ]:
# ── Cell 15: Save Checkpoint Back to Drive ─────────────────────────────────
import shutil

DRIVE_CKPT = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

for fname in os.listdir('checkpoints'):
    src = os.path.join('checkpoints', fname)
    dst = os.path.join(DRIVE_CKPT, fname)
    shutil.copy2(src, dst)
    print(f'Saved {fname} → Drive')

print('All checkpoints saved to Google Drive.')